### Workshop ➜ DLT example.

In [8]:
! uv add dlt[duckdb]
import dlt
from dlt.sources.rest_api import rest_api_source

Resolved 161 packages in 5ms
Audited 156 packages in 3ms


---

#### Define API source.

In [5]:
def openlibrary_source(query: str = "harry potter"):

    return rest_api_source({
        "client": {
            "base_url": "https://openlibrary.org",
        },
        "resource_defaults": {
            "primary_key": "key",
            "write_disposition": "replace",
        },
        "resources": [
            {
                "name": "books",
                "endpoint": {
                    "path": "search.json",
                    "params": {
                        "q": query,
                        "limit": 100,
                    },
                    "data_selector": "docs",
                    "paginator": {
                        "type": "offset",
                        "limit": 100,
                        "offset_param": "offset",
                        "limit_param": "limit",
                        "total_path": "numFound",
                    },
                },
            },
        ],
    })


---

#### Pipeline init.

In [6]:
pipeline = dlt.pipeline(
    pipeline_name="ol_demo",
    destination="duckdb",
    dataset_name="ol_data",
    progress="log" 
)

---

This is an alternative way to run all ETL steps at once:

In [14]:
# source = openlibrary_source(query="harry potter")
# pipeline.run(source)

---

#### Extraction.

In [7]:
extract_info = pipeline.extract(openlibrary_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 178.33 MB (59.60%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.83s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 9986438.10/s
Memory usage: 182.08 MB (59.50%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 2.17s | Rate: 0.00/s
books: 400  | Time: 1.34s | Rate: 298.71/s
Memory usage: 183.17 MB (59.50%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 3.89s | Rate: 0.00/s
books: 700  | Time: 3.06s | Rate: 229.09/s
Memory usage: 184.67 MB (59.50%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 5.21s | Rate: 0.

In [5]:
load_id = extract_info.loads_ids[-1]
m = extract_info.metrics[load_id][0]

print("Resources:", list(m["resource_metrics"].keys()))
print("Tables:", list(m["table_metrics"].keys()))
print("Load ID:", load_id)
print()

for resource, rm in m["resource_metrics"].items():
    print(f"Resource: {resource}")
    print(f"rows extracted: {rm.items_count}")
    print()

Resources: ['books']
Tables: ['books']
Load ID: 1772308737.411977

Resource: books
rows extracted: 3759



---

#### Normalization.

In [6]:
normalize_info = pipeline.normalize()

------------------- Normalize rest_api in 1772308737.411977 --------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 160.38 MB (57.30%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772308737.411977 --------------------
Files: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Items: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 160.38 MB (57.30%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1772308737.411977 --------------------
Files: 9/1 (900.0%) | Time: 0.28s | Rate: 32.17/s
Items: 23082  | Time: 0.28s | Rate: 82581.14/s
Memory usage: 171.14 MB (57.30%) | CPU usage: 0.00%



In [7]:
load_id = normalize_info.loads_ids[-1]
m = normalize_info.metrics[load_id][0]

print("Load ID:", load_id)
print()

print("Tables created/updated:")
for table_name, tm in m["table_metrics"].items():
    # skip dlt internal tables to keep it beginner-friendly
    if table_name.startswith("_dlt"):
        continue
    print(f"  - {table_name}: {tm.items_count} rows")

Load ID: 1772308737.411977

Tables created/updated:
  - books: 3759 rows
  - books__author_key: 4637 rows
  - books__author_name: 4637 rows
  - books__ia: 3434 rows
  - books__ia_collection: 2735 rows
  - books__language: 3748 rows
  - books__id_standard_ebooks: 12 rows
  - books__id_librivox: 64 rows
  - books__id_project_gutenberg: 56 rows


---

#### Load.

In [8]:
load_info = pipeline.load()

---------------------- Load rest_api in 1772308737.411977 ----------------------
Jobs: 0/9 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 184.09 MB (57.30%) | CPU usage: 0.00%

---------------------- Load rest_api in 1772308737.411977 ----------------------
Jobs: 9/9 (100.0%) | Time: 0.47s | Rate: 19.08/s
Memory usage: 234.62 MB (57.30%) | CPU usage: 0.00%



---

#### Run full pipeline.

In [9]:
load_info = pipeline.run(openlibrary_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 234.12 MB (57.30%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.53s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 16131938.46/s
Memory usage: 234.17 MB (57.30%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 1.76s | Rate: 0.00/s
books: 600  | Time: 1.23s | Rate: 488.01/s
Memory usage: 234.17 MB (57.30%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 2.88s | Rate: 0.00/s
books: 1000  | Time: 2.35s | Rate: 425.99/s
Memory usage: 234.17 MB (57.30%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 4.06s | Rate: 

---

#### Data inspection.

In [11]:
ds = pipeline.dataset()
ds.tables

['books',
 'books__author_key',
 'books__author_name',
 'books__ia',
 'books__ia_collection',
 'books__language',
 'books__id_standard_ebooks',
 'books__id_librivox',
 'books__id_project_gutenberg',
 '_dlt_version',
 '_dlt_loads',
 '_dlt_pipeline_state']

In [12]:
df = ds.books.df()  
df.head(3)

,cover_edition_key,cover_i,ebook_access,edition_count,first_publish_year,has_fulltext,key,lending_edition_s,lending_identifier_s,public_scan_b,title,_dlt_load_id,_dlt_id,subtitle
0,OL61027601M,15155833,borrowable,397,1997,True,/works/OL82563W,OL38565767M,harrypotterylapi0000rowl_q5r6,False,Harry Potter and the Philosopher's Stone,1772308755.304492,EFzuAVYjKaM1AQ,NaN
1,OL26378158M,15158660,printdisabled,144,2007,True,/works/OL82586W,NaN,NaN,False,Harry Potter and the Deathly Hallows,1772308755.304492,WlmCuHAdA7OTKw,NaN
2,OL26234270M,10580435,borrowable,279,1999,True,/works/OL82536W,OL48101764M,bdrc-W8LS66814,False,Harry Potter and the Prisoner of Azkaban,1772308755.304492,5CPXVAeNgb2Daw,NaN
